# Smoking is bad

In [ ]:
# TODO: install all the necessary packages
# Packages are managed by Pixi, not pip: everything is listed in pixi.toml.
# From the repo root run `pixi install`, then `pixi run lab` (or pick the `smoke`
# environment's Python as the kernel in your IDE). Here we only import them.
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

# Two fixed colours used in every plot: blue = non-smokers, orange = smokers
COLOURS = {"Non-smoker": "#2a78d6", "Smoker": "#eb6834"}

In [ ]:
# TODO: load the data using the pathlib library to create the path to your data
DATA_PATH = Path.cwd() / "data" / "smoking_data.csv"
df_raw = pd.read_csv(DATA_PATH, index_col=0)
df_raw.head()

In [ ]:
# TODO: what are the dimensions of the data frame?
n_rows, n_cols = df_raw.shape
print(f"{n_rows} rows (people) and {n_cols} columns: {list(df_raw.columns)}")

In [ ]:
# TODO: use .describe() on your data frame: what do you notice about the string columns?
df_raw.describe()

In [ ]:
# By default .describe() silently drops the string columns. Ask for them explicitly:
df_raw.describe(include="all")

**What we notice:** plain `.describe()` only summarises the numeric columns (`age`, `salary`).
`outcome`, `smoker` and `gender` vanish because they are stored as text. With
`include="all"` they only get `count`, `unique`, `top` and `freq`: each has exactly two values,
so they are really binary variables. To take a mean (a survival rate, a smoking rate) we need to
turn them into 0/1 numbers first.

In [ ]:
# TODO: do not forget to check the null values
nulls = df_raw.isna().sum().to_frame("n_missing")
nulls["share_missing"] = (nulls["n_missing"] / len(df_raw)).round(3)
nulls

In [ ]:
# Is salary missing at random, or are the missing rows a different kind of person?
df_raw.assign(salary_missing=df_raw["salary"].isna()).groupby("salary_missing").agg(
    n=("age", "size"),
    mean_age=("age", "mean"),
    share_alive=("outcome", lambda s: (s == "Alive").mean()),
    share_smokers=("smoker", lambda s: (s == "Yes").mean()),
).round(3)

Only `salary` has gaps, and a lot of them: 700 of 1,314 rows (53%). The people with and without a
salary look alike on age, survival and smoking, so the gaps seem random. We keep every row and
only use `salary` where it exists, rather than dropping half the sample or inventing values.

In [ ]:
# TODO: prepare your data frame so that you can work with
AGE_BINS = [17, 24, 34, 44, 54, 64, 74, 100]
AGE_LABELS = ["18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75+"]

df = df_raw.assign(
    alive=(df_raw["outcome"] == "Alive").astype(int),
    smokes=(df_raw["smoker"] == "Yes").astype(int),
    male=(df_raw["gender"] == "male").astype(int),
    status=np.where(df_raw["smoker"] == "Yes", "Smoker", "Non-smoker"),
    age_group=pd.cut(df_raw["age"], bins=AGE_BINS, labels=AGE_LABELS),
)
df.head()

## Smoking analysis
Let's first check what your cousin did.

In [ ]:
# TODO: Reimplement the small code snippet described in the problem set instructions.
df.groupby(["smokes"]).agg(prob=("alive", np.mean)).round(3)

You should get the same results.

# Is smoking really good?
Now it is up to you to run the analysis further and check what could be underlying your cousings results.

The plan: (1) redraw the cousin's result, (2) check whether smokers and non-smokers are the same
kind of people, (3) compare like with like, (4) put a number on it with a regression.

## 1. The cousin's result, as a picture

In [ ]:
headline = df.groupby("status")["alive"].agg(["mean", "size"]).reindex(["Non-smoker", "Smoker"])

fig, ax = plt.subplots(figsize=(5, 3.5))
bars = ax.bar(headline.index, headline["mean"], color=[COLOURS[s] for s in headline.index], width=0.6)
for bar, (rate, n) in zip(bars, headline.itertuples(index=False)):
    ax.text(bar.get_x() + bar.get_width() / 2, rate + 0.02, f"{rate:.1%}\n(n={n})", ha="center", va="bottom")
ax.set_ylim(0, 1)
ax.set_ylabel("Share alive at follow-up")
ax.set_title("Raw comparison: smokers survive more often")
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
plt.tight_layout()
plt.show()

The 7.5 point gap is real. The question is whether smoking causes it, or whether smokers and
non-smokers differ in some other way that also affects survival. Of the variables we have,
age is the obvious suspect: it has a huge effect on whether someone is still alive later.

## 2. Are smokers and non-smokers the same kind of people?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

# Left: age distribution by smoking status
sns.histplot(
    data=df, x="age", hue="status", hue_order=list(COLOURS), palette=COLOURS,
    bins=range(18, 90, 5), stat="density", common_norm=False, element="step", fill=False, linewidth=2,
    ax=axes[0],
)
for status, colour in COLOURS.items():
    axes[0].axvline(df.loc[df["status"] == status, "age"].mean(), color=colour, linestyle="--", linewidth=1.5)
axes[0].set_title("Age distribution (dashed = mean)")
axes[0].set_xlabel("Age at start of study")
axes[0].get_legend().set_title(None)

# Right: share of each age group that smokes
smoke_rate = df.groupby("age_group", observed=True)["smokes"].agg(["mean", "size"])
axes[1].bar(smoke_rate.index.astype(str), smoke_rate["mean"], color=COLOURS["Smoker"], width=0.6)
for i, (rate, n) in enumerate(smoke_rate.itertuples(index=False)):
    axes[1].text(i, rate + 0.015, f"{rate:.0%}", ha="center", va="bottom", fontsize=9)
axes[1].set_ylim(0, 0.75)
axes[1].set_title("Share who smoke, by age group")
axes[1].set_xlabel("Age group")
axes[1].set_ylabel("Share smoking")
axes[1].yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))

plt.tight_layout()
plt.show()

df.groupby("status")["age"].describe().round(1)

**Smokers are younger.** Their mean age is about 4 years lower, and the gap is concentrated at the
top: only 22% of people aged 65-74 and 17% of those 75+ smoke, against roughly half of everyone
under 65. The non-smoker group is loaded with old people.

In [ ]:
survival_by_age = df.groupby("age_group", observed=True)["alive"].agg(["mean", "size"])

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(survival_by_age.index.astype(str), survival_by_age["mean"], color="#555555", marker="o", markersize=8, linewidth=2)
for i, (rate, n) in enumerate(survival_by_age.itertuples(index=False)):
    ax.annotate(f"{rate:.0%}", (i, rate), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=9)
ax.set_ylim(-0.05, 1.1)
ax.set_title("Survival falls steeply with age (everyone)")
ax.set_xlabel("Age group")
ax.set_ylabel("Share alive at follow-up")
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
plt.tight_layout()
plt.show()

**Age drives survival.** Almost everyone under 35 is alive at follow-up; almost nobody over 75 is.
So age passes both tests of a confounder: it is linked to smoking (old people smoke less) and to
the outcome (old people die more). That alone can make smokers look healthier, even if smoking
does nothing or does harm.

## 3. Compare like with like: survival within each age group

In [ ]:
strat = (
    df.groupby(["age_group", "status"], observed=True)["alive"]
    .agg(["mean", "size"])
    .reset_index()
)

fig, ax = plt.subplots(figsize=(8, 4.2))
for status, colour in COLOURS.items():
    sub = strat[strat["status"] == status]
    ax.plot(sub["age_group"].astype(str), sub["mean"], color=colour, marker="o", markersize=8, linewidth=2, label=status)
ax.set_ylim(-0.05, 1.08)
ax.set_title("Same age: smokers survive less, or equally, in every group")
ax.set_xlabel("Age group")
ax.set_ylabel("Share alive at follow-up")
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

table = strat.pivot(index="age_group", columns="status", values=["mean", "size"])
table.columns = [f"{'alive' if a == 'mean' else 'n'} {b.lower()}" for a, b in table.columns]
table["gap (smoker - non)"] = table["alive smoker"] - table["alive non-smoker"]
table.round(3)

Once we compare people of the same age, the sign flips. In five of the seven age groups smokers
survive less often, in one they are level (25-34), and in 75+ everyone is dead either way. This is
**Simpson's paradox**: a pattern in the pooled data that reverses inside every subgroup, because
the subgroups are mixed in different proportions.

One number to summarise it: **age-standardised survival**. Give both groups the same age mix
(the whole sample's) and recompute their survival rates.

In [ ]:
age_weights = df["age_group"].value_counts(normalize=True).sort_index()
rates = df.pivot_table(index="age_group", columns="status", values="alive", aggfunc="mean", observed=True)

standardised = pd.DataFrame(
    {
        "raw survival": df.groupby("status")["alive"].mean(),
        "age-standardised survival": rates.mul(age_weights, axis=0).sum(),
    }
).reindex(list(COLOURS))
standardised.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.8))
x = np.arange(len(standardised.columns))
width = 0.36
for i, status in enumerate(COLOURS):
    vals = standardised.loc[status]
    bars = ax.bar(x + (i - 0.5) * width, vals, width=width * 0.94, color=COLOURS[status], label=status)
    ax.bar_label(bars, labels=[f"{v:.1%}" for v in vals], padding=3, fontsize=9)
ax.set_xticks(x, standardised.columns)
ax.set_ylim(0, 0.95)
ax.set_ylabel("Share alive at follow-up")
ax.set_title("Same age mix, opposite conclusion")
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
ax.legend(frameon=False, loc="upper right")
plt.tight_layout()
plt.show()

Raw: smokers 76% vs non-smokers 69%. Same age mix: smokers 70% vs non-smokers 74%.
The cousin's advantage disappears and turns into a disadvantage.

## 4. The other variables: gender and salary

Age is not the only column. Could gender or salary be doing the work too?

In [ ]:
gender_age = df.groupby(["gender", "status"])["age"].mean().unstack().round(1)
gender_rates = df.groupby(["gender", "status"])["alive"].mean().unstack().round(3)
display(gender_age.add_prefix("mean age, "))
display(gender_rates.add_prefix("share alive, "))

strat_g = (
    df.groupby(["gender", "age_group", "status"], observed=True)["alive"]
    .agg(["mean", "size"])
    .reset_index()
)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, gender in zip(axes, ["female", "male"]):
    for status, colour in COLOURS.items():
        sub = strat_g[(strat_g["gender"] == gender) & (strat_g["status"] == status)]
        ax.plot(sub["age_group"].astype(str), sub["mean"], color=colour, marker="o", markersize=7, linewidth=2, label=status)
    ax.set_title(f"{gender.capitalize()}s")
    ax.set_xlabel("Age group")
    ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
axes[0].set_ylabel("Share alive at follow-up")
axes[0].legend(frameon=False)
fig.suptitle("Survival by age group, split by gender", y=1.02)
plt.tight_layout()
plt.show()

**Gender is not the driver, but it shows where the age effect hides.** Among women, smokers and
non-smokers are the same age on average and the raw gap is small. Among men, non-smokers are about
7 years older than smokers, and that is exactly where the raw "smokers survive more" gap is biggest
(75% vs 64%). Split by age, the smoker line sits at or below the non-smoker line almost everywhere for
both genders. The one exception, women aged 65-74, rests on a handful of smokers.

In [ ]:
with_salary = df.dropna(subset=["salary"]).assign(
    salary_quartile=lambda d: pd.qcut(d["salary"], 4, labels=["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"])
)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

sns.boxplot(
    data=with_salary, x="status", y="salary", hue="status", palette=COLOURS, order=list(COLOURS),
    width=0.5, saturation=1, legend=False, ax=axes[0],
)
axes[0].set_title("Salary is similar for smokers and non-smokers")
axes[0].set_xlabel(None)
axes[0].set_ylabel("Salary")

q = with_salary.groupby("salary_quartile", observed=True)["alive"].mean()
axes[1].bar(q.index.astype(str), q.values, color="#888888", width=0.6)
for i, v in enumerate(q.values):
    axes[1].text(i, v + 0.015, f"{v:.0%}", ha="center", va="bottom", fontsize=9)
axes[1].set_ylim(0, 1)
axes[1].set_title("Survival barely moves with salary")
axes[1].set_xlabel("Salary quartile")
axes[1].set_ylabel("Share alive at follow-up")
axes[1].yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))

plt.tight_layout()
plt.show()

with_salary[["salary", "age", "smokes", "alive"]].corr().round(3)

**Salary is a non-factor here.** Smokers and non-smokers earn about the same, survival is flat
across salary quartiles, and salary is essentially uncorrelated with age, smoking or survival.
It cannot explain the cousin's result (and with half the values missing, it is a weak variable anyway).

## 5. Putting a number on it: logistic regression

Same story in one table. Each row is a model of the chance of being alive; the column shows
the odds ratio on smoking (above 1 = smokers more likely to be alive, below 1 = less likely).

In [ ]:
specs = {
    "smoking only": "alive ~ smokes",
    "+ age": "alive ~ smokes + age",
    "+ age + gender": "alive ~ smokes + age + male",
    "+ age groups + gender": "alive ~ smokes + C(age_group_reg) + male",
    "+ age + gender + salary": "alive ~ smokes + age + male + salary",
}

# For the age-group model, merge 65-74 and 75+: nobody aged 75+ survived, which makes
# a separate 75+ dummy impossible to estimate (perfect separation).
df["age_group_reg"] = pd.cut(df["age"], bins=[17, 24, 34, 44, 54, 64, 100], labels=["18-24", "25-34", "35-44", "45-54", "55-64", "65+"])

rows = []
for name, formula in specs.items():
    data = df.dropna(subset=["salary"]) if "salary" in formula else df
    fit = smf.logit(formula, data=data).fit(disp=0)
    lo, hi = np.exp(fit.conf_int().loc["smokes"])
    rows.append(
        {"model": name, "odds ratio (smokes)": np.exp(fit.params["smokes"]), "95% CI low": lo,
         "95% CI high": hi, "p-value": fit.pvalues["smokes"], "n": int(fit.nobs)}
    )
results = pd.DataFrame(rows).set_index("model")
results.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
y = np.arange(len(results))[::-1]
ax.errorbar(
    results["odds ratio (smokes)"], y,
    xerr=[results["odds ratio (smokes)"] - results["95% CI low"], results["95% CI high"] - results["odds ratio (smokes)"]],
    fmt="none", ecolor="#999999", elinewidth=2, capsize=4,
)
ax.scatter(results["odds ratio (smokes)"], y, s=70, color="#333333", zorder=3)
ax.axvline(1, color="#444444", linestyle="--", linewidth=1)
ax.set_yticks(y, results.index)
ax.set_xscale("log")
ax.set_xticks([0.5, 0.75, 1, 1.5, 2], ["0.5", "0.75", "1", "1.5", "2"])
ax.xaxis.set_minor_formatter(plt.matplotlib.ticker.NullFormatter())
ax.set_xlabel("Odds ratio of being alive for smokers (log scale, 95% CI)")
ax.set_title("Adding age flips the smoking effect below 1")
plt.tight_layout()
plt.show()

Without controls, smokers have about 1.46 times the odds of being alive (p = 0.003). Add age and the
odds ratio drops below 1: smokers are now *less* likely to be alive. Adding gender or salary
changes almost nothing, which confirms age is the confounder.

How sure can we be? It depends on how age enters the model. A straight line in age gives an odds
ratio of about 0.81 (p = 0.22, not significant). Age groups, which fit the very non-linear
survival curve better, give about 0.67 (p = 0.025). Either way, the data give no support at all to
"smoking helps", and the better-fitting model says it hurts.

# Conclusion

- **Your cousin's numbers are right; her reading of them is wrong.** Smokers in this sample do
  survive more often (76% vs 69%).
- **Age explains it.** Non-smokers are older on average, and far more of the over-65s are
  non-smokers. Old people die more, whatever they smoke, which drags the non-smoker rate down.
- **Compare people of the same age and the effect reverses.** Smokers survive less in 5 of 7 age
  groups; age-standardised survival is 70% for smokers vs 74% for non-smokers; the regression
  odds ratio falls from 1.46 to between 0.67 and 0.81 depending on how age is modelled.
- **Gender and salary are not the drivers.** Gender only matters because male non-smokers are
  older; salary has no relationship with smoking or survival.
- **Caveat:** this is observational data with only a few variables. Other things we cannot see
  (health at the start, alcohol, occupation) could still matter, and the size of the harm is
  estimated loosely. The defensible claim: this data does not show smoking helps, and once age is
  handled it points the other way, in line with the wider medical evidence.